# interactive oscillator

playing with `ipywidgets` so i can *drag* the parameters instead of editing the file and rerunning. the damped driven oscillator is perfect for this -- slide the drive frequency toward the natural one and watch resonance kick in.

$\ddot{x} + \frac{c}{m}\dot{x} + \frac{k}{m}x = F_0\cos(\omega_d t)$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

k, m, w0 = 4.0, 1.0, 2.0   # natural freq sqrt(k/m) = 2

def response(c=0.5, F0=1.0, wd=1.5):
    def deriv(s, t):
        x, v = s
        return np.array([v, (-k*x - c*v + F0*np.cos(wd*t))/m])
    dt, n = 0.01, 4000
    s = np.array([0.0, 0.0]); X = np.zeros(n); T = np.arange(n)*dt
    for i in range(n):
        t = i*dt
        k1 = deriv(s, t); k2 = deriv(s+0.5*dt*k1, t+0.5*dt)
        k3 = deriv(s+0.5*dt*k2, t+0.5*dt); k4 = deriv(s+dt*k3, t+dt)
        s = s + (dt/6)*(k1+2*k2+2*k3+k4); X[i] = s[0]
    plt.figure(figsize=(9,3.5))
    plt.plot(T, X)
    plt.title(f'drive wd={wd:.2f} (resonance at {w0})'); plt.xlabel('t'); plt.grid(True)
    plt.show()

interact(response,
         c=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.4),
         F0=FloatSlider(min=0.5, max=3.0, step=0.5, value=1.0),
         wd=FloatSlider(min=0.5, max=3.5, step=0.1, value=1.5));

interactive(children=(FloatSlider(value=0.4, description='c', max=2.0, min=0.1), FloatSlider(value=1.0, descri...

and because i cant resist, here's the same thing as a gif -- sweeping the drive frequency up through resonance. the swing balloons right at $\omega_d=\omega_0=2$.

In [2]:
from matplotlib.animation import FuncAnimation, PillowWriter
import os
os.makedirs('media', exist_ok=True)

def steady(wd, c=0.4, F0=1.0):
    def deriv(s, t):
        x, v = s
        return np.array([v, (-k*x - c*v + F0*np.cos(wd*t))/m])
    dt, n = 0.01, 4000
    s = np.array([0.0, 0.0]); X = np.zeros(n); T = np.arange(n)*dt
    for i in range(n):
        t = i*dt
        k1=deriv(s,t); k2=deriv(s+0.5*dt*k1,t+0.5*dt)
        k3=deriv(s+0.5*dt*k2,t+0.5*dt); k4=deriv(s+dt*k3,t+dt)
        s = s+(dt/6)*(k1+2*k2+2*k3+k4); X[i]=s[0]
    return T, X

wds = np.linspace(0.6, 3.2, 60)
fig, ax = plt.subplots(figsize=(9,3.5))
line, = ax.plot([], [])
ax.set_xlim(0, 40); ax.set_ylim(-6, 6); ax.set_xlabel('t'); ax.grid(True)
def update(i):
    T, X = steady(wds[i])
    line.set_data(T, X)
    ax.set_title(f'drive wd = {wds[i]:.2f}  (resonance at 2.0)')
    return [line]
anim = FuncAnimation(fig, update, frames=len(wds), blit=False, interval=80)
anim.save('media/oscillator_sweep.gif', writer=PillowWriter(fps=15))
plt.close(fig)
print('saved media/oscillator_sweep.gif')

saved media/oscillator_sweep.gif
